# Fundamental Ranking Lab — Cloud Pilot Runner (Google Colab)

Notebook này thực hiện tính toán ma trận 8 chỉ số tài chính Point-in-Time và xuất báo cáo khả thi (Stage 01 Pilot) trên môi trường Google Colab.

### Quy trình thực hiện:
1. **Mount Google Drive** để liên kết kho lưu trữ dữ liệu thô.
2. **Giải nén bộ dữ liệu thô** (`pilot_raw_bundle.zip`) vào môi trường SSD cục bộ của Colab.
3. **Cài đặt môi trường** và các thư viện cần thiết với `uv`.
4. **Chạy pipeline tính toán** (`uv run frl pilot`).
5. **Xuất kết quả Parquet và báo cáo** ngược lại vào Google Drive.

## 1. Kết nối Google Drive & Cấu hình bảo mật

In [ ]:
import os
import pathlib
import shutil

from google.colab import drive, userdata

# Mount Google Drive
drive.mount('/content/drive')

# Thiết lập User-Agent từ Colab Secrets (biểu tượng Chìa khóa bên trái)
try:
    sec_contact = userdata.get('FRL_SEC_USER_AGENT')
    os.environ['FRL_SEC_USER_AGENT'] = sec_contact
    print('Đã nạp thành công FRL_SEC_USER_AGENT từ Secrets!')
except Exception:
    print('Chưa cấu hình secret FRL_SEC_USER_AGENT trong Colab Secrets!')
    print('Vui lòng mở biểu tượng Chìa khóa (🔑) bên trái và thêm: FRL_SEC_USER_AGENT = Tên và email của bạn.')

## 2. Clone mã nguồn & Cài đặt môi trường với `uv`

In [ ]:
# Cài đặt uv (trình quản lý gói Python tốc độ cao)
!pip install -q uv

# Kiểm tra mã nguồn dự án
project_dir = pathlib.Path('/content/fundamental-ranking-lab')
if not project_dir.exists():
    !git clone https://github.com/ryantr-statinops/fundamental-ranking-lab.git /content/fundamental-ranking-lab

%cd /content/fundamental-ranking-lab
!git pull

# Kiểm tra môi trường
!uv run frl doctor

## 3. Giải nén dữ liệu thô từ Google Drive vào Colab SSD

In [ ]:
# Đường dẫn file nén trên Google Drive
drive_bundle = pathlib.Path('/content/drive/MyDrive/FundamentalRankingLab/pilot_raw_bundle.zip')
local_raw_dir = pathlib.Path('/content/fundamental-ranking-lab/data/raw')

if drive_bundle.exists():
    print(f'Tìm thấy file dữ liệu: {drive_bundle}')
    local_raw_dir.mkdir(parents=True, exist_ok=True)
    print('Đang giải nén dữ liệu vào Colab SSD...')
    !unzip -q -o "{drive_bundle}" -d /content/fundamental-ranking-lab/data/raw/
    print('Giải nén hoàn tất! Sẵn sàng tính toán.')
else:
    print(f'Không tìm thấy file: {drive_bundle}')
    print('Vui lòng đảm bảo bạn đã tạo thư mục FundamentalRankingLab trên Drive và upload file pilot_raw_bundle.zip vào đó.')

## 4. Chạy Pipeline bóc tách & Tính toán chỉ số tài chính (Point-in-Time Pilot)

In [ ]:
# Kích hoạt pipeline tính toán với realtime streaming log (PYTHONUNBUFFERED=1)
!PYTHONUNBUFFERED=1 uv run frl pilot --config configs/pilot.toml

## 5. Sao lưu kết quả Parquet & Báo cáo ngược lại Google Drive

In [ ]:
# Tìm artifact run mới nhất
artifacts_dir = pathlib.Path('/content/fundamental-ranking-lab/artifacts')
runs = [d for d in artifacts_dir.iterdir() if d.is_dir() and not d.name.startswith('.')]
if runs:
    latest_run = max(runs, key=lambda d: d.stat().st_mtime)
    print(f'Run artifact mới nhất: {latest_run.name}')
    
    # Thư mục đích trên Drive
    dest_drive = pathlib.Path('/content/drive/MyDrive/FundamentalRankingLab/results') / latest_run.name
    dest_drive.mkdir(parents=True, exist_ok=True)
    
    # Sao chép các file kết quả quan trọng
    for f in latest_run.iterdir():
        if f.is_file() and (f.suffix in ['.parquet', '.json', '.md', '.html', '.csv']):
            shutil.copy2(f, dest_drive / f.name)
            print(f'  + Đã lưu file: {f.name} -> Drive')
            
    print(f'\nToàn bộ kết quả đã được sao lưu an toàn tại: Google Drive/FundamentalRankingLab/results/{latest_run.name}/')
else:
    print('Chưa có artifact nào được sinh ra.')